# 5-Fold ModernNCA + FT-Transformer + RealMLP (Kaggle GPU)

**Cấu hình chạy trên Kaggle:** Accelerator = GPU T4 x2, Internet = On (cài `pytabkit`).
- Chia chéo 5-Fold (`StratifiedKFold`, `seed=42`) trên `train.csv`.
- Gồm 3 kiến trúc: `RealMLP_TD_Classifier`, `ModernNCA` (truy hồi láng giềng sâu), và `FT-Transformer` (24 trường).
- Kết quả xuất ra `/kaggle/working/`: `submission.csv`, `oof_kaggle_sota2025_v2_5fold.csv`, `sub_kaggle_sota2025_v2_details.csv`.


In [ ]:
!pip install -q pytabkit
import torch
try:
    import pytabkit
    print('pytabkit:', getattr(pytabkit, '__version__', 'installed'))
except Exception as e:
    print('pytabkit not installed (will use pure-PyTorch Category-Embedding RealMLP fallback):', e)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
"""Quy trình 5-Fold trên Kaggle GPU gồm 3 kiến trúc: RealMLP, ModernNCA và FT-Transformer 24 trường."""

from __future__ import annotations

import gc
import math
import os
from pathlib import Path
import time

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.linear_model import Ridge
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch
from torch import nn
import torch.nn.functional as F

try:
    from pytabkit import RealMLP_TD_Classifier
    HAS_PYTABKIT = True
except Exception:
    RealMLP_TD_Classifier = None
    HAS_PYTABKIT = False

# Cấu hình huấn luyện trên Kaggle GPU T4
DRY_RUN = os.environ.get("KAGGLE_DRY_RUN", "0") == "1"
SEED = 42
N_SPLITS = 2 if DRY_RUN else 5

REALMLP_EPOCHS = 2 if DRY_RUN else 6
REALMLP_BATCH = 256

NCA_EPOCHS = 2 if DRY_RUN else 12
NCA_PATIENCE = 3
NCA_BATCH = 256 if DRY_RUN else 2048
NCA_TRAIN_POS_SUP = 128 if DRY_RUN else 512
NCA_TRAIN_NEG_SUP = 384 if DRY_RUN else 1536
NCA_EVAL_NEG_SUP = 800 if DRY_RUN else 48000  # cộng toàn bộ ca gian lận thật mỗi fold

FTT_EPOCHS = 2 if DRY_RUN else 14
FTT_PATIENCE = 4
FTT_BATCH = 256 if DRY_RUN else 2048

ID_COLUMN = "id"
TARGET_COLUMN = "is_fraud"
NUMERIC_COLUMNS = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
]
FLOAT_COLUMNS = [
    "amount_usd",
    "account_balance_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "velocity_score",
    "merchant_risk_score",
]
BINARY_COLUMNS = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]
CATEGORICAL_COLUMNS = [
    "merchant_category",
    "card_type",
    "auth_method",
    "channel",
    "device_type",
]


def locate_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/playground-series-s6e11"),
        Path("/kaggle/input/competitions/playground-series-s6e11"),
        Path("dataset/playground-series-s6e11"),
        Path("../dataset/playground-series-s6e11"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for p in kaggle_root.rglob("train.csv"):
            if (p.parent / "test.csv").exists():
                return p.parent
    raise FileNotFoundError("Could not find train.csv / test.csv.")


def locate_output_dir() -> Path:
    custom = os.environ.get("KAGGLE_OUT_DIR")
    if custom:
        out = Path(custom)
        out.mkdir(parents=True, exist_ok=True)
        return out
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    out = Path("artifacts/kaggle_sota2025_v2_dryrun") if DRY_RUN else Path("artifacts/kaggle_sota2025_v2")
    out.mkdir(parents=True, exist_ok=True)
    return out


def safe_logit(p: np.ndarray | float) -> np.ndarray | float:
    return logit(np.clip(p, 1e-6, 1.0 - 1e-6))


def _leading_digits(int_arr: np.ndarray, k: int) -> np.ndarray:
    vals = np.maximum(int_arr, 0).astype(np.int64)
    out = np.zeros_like(vals)
    pos = vals > 0
    if not pos.any():
        return out
    v = vals[pos]
    digits = np.floor(np.log10(v.astype(np.float64))).astype(np.int64) + 1
    shift = np.maximum(digits - k, 0)
    pow10 = np.power(10, np.minimum(shift, 12), dtype=np.int64)
    out[pos] = v // pow10
    return out


# Phần 1: Đặc trưng phân loại thuần cho RealMLP (không dùng Target Encoding)
def build_realmlp_no_te_tables(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Tạo bảng đặc trưng số, chữ số và phân loại cho RealMLP (không dùng Target Encoding)."""
    frames = []
    for df in (train_df, test_df):
        d: dict[str, np.ndarray | pd.Series] = {}
        # 1. Biến số gốc, cờ nhị phân và đặc trưng nghiệp vụ
        for c in NUMERIC_COLUMNS:
            d[c] = df[c].to_numpy(dtype=np.float32)
        for c in BINARY_COLUMNS:
            d[c] = df[c].to_numpy(dtype=np.float32)

        amt = df["amount_usd"].to_numpy(dtype=np.float64)
        bal = df["account_balance_usd"].to_numpy(dtype=np.float64)
        hrs = df["hours_since_last_txn"].to_numpy(dtype=np.float64)
        dist = df["distance_from_home_km"].to_numpy(dtype=np.float64)
        hour = df["time_of_day_hour"].to_numpy(dtype=np.float64)
        dow = df["day_of_week"].to_numpy(dtype=np.float64)

        d["flag_sum"] = df[BINARY_COLUMNS].sum(axis=1).to_numpy(dtype=np.float32)
        d["log_amt"] = np.log1p(np.maximum(amt, 0.0)).astype(np.float32)
        d["log_bal"] = np.log1p(np.maximum(bal, 0.0)).astype(np.float32)
        d["log_hrs"] = np.log1p(np.maximum(hrs, 0.0)).astype(np.float32)
        d["log_dist"] = np.log1p(np.maximum(dist, 0.0)).astype(np.float32)
        d["amt_over_bal"] = (amt / (np.maximum(bal, 0.0) + 1.0)).astype(np.float32)
        d["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0).astype(np.float32)
        d["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0).astype(np.float32)
        d["dow_sin"] = np.sin(2.0 * np.pi * dow / 7.0).astype(np.float32)
        d["dow_cos"] = np.cos(2.0 * np.pi * dow / 7.0).astype(np.float32)

        # 2. Chữ số phần lẻ dạng số nguyên
        for c in FLOAT_COLUMNS:
            c100 = np.rint(df[c].to_numpy(dtype=np.float64) * 100.0).astype(np.int64)
            ip = np.floor_divide(c100, 100)
            d[f"{c}__dec2_num"] = np.mod(c100, 100).astype(np.float32)
            d[f"{c}__cent1_num"] = np.mod(c100, 10).astype(np.float32)
            d[f"{c}__cent2_num"] = np.mod(np.floor_divide(c100, 10), 10).astype(np.float32)
            d[f"{c}__unit_num"] = np.mod(ip, 10).astype(np.float32)
            d[f"{c}__tens_num"] = np.mod(np.floor_divide(ip, 10), 10).astype(np.float32)
            d[f"{c}__lead1_num"] = _leading_digits(ip, 1).astype(np.float32)

        # 3. Cột phân loại (gốc, giá trị lặp lại và chữ số thập phân)
        cat_cols: list[str] = []
        for c in CATEGORICAL_COLUMNS:
            d[c] = df[c].astype(str)
            cat_cols.append(c)
        for c in NUMERIC_COLUMNS:
            c100 = np.rint(df[c].to_numpy(dtype=np.float64) * 100.0).astype(np.int64)
            cn = f"cat_ex__{c}"
            d[cn] = c100
            cat_cols.append(cn)
        for c in FLOAT_COLUMNS:
            c100 = np.rint(df[c].to_numpy(dtype=np.float64) * 100.0).astype(np.int64)
            ip = np.floor_divide(c100, 100)
            for suf, arr in [
                ("int", ip),
                ("r1", np.floor_divide(c100, 10)),
                ("dec2", np.mod(c100, 100)),
                ("cent1", np.mod(c100, 10)),
                ("cent2", np.mod(np.floor_divide(c100, 10), 10)),
                ("lead2", _leading_digits(ip, 2)),
            ]:
                cn = f"cat_{suf}__{c}"
                d[cn] = arr
                cat_cols.append(cn)
        frames.append(pd.DataFrame(d))

    return frames[0], frames[1], cat_cols


def encode_fold_categories(
    X_tr: pd.DataFrame,
    X_va: pd.DataFrame,
    X_te: pd.DataFrame,
    cat_cols: list[str],
    min_count: int = 2,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[int]]:
    """Mã hóa cột phân loại theo từng fold: gán giá trị hiếm hoặc chưa gặp về chỉ số 0."""
    tr_out = X_tr.copy()
    va_out = X_va.copy()
    te_out = X_te.copy()
    card_sizes: list[int] = []

    for c in cat_cols:
        if c in CATEGORICAL_COLUMNS:
            vals = sorted(tr_out[c].unique().tolist())
            mp = {v: i + 1 for i, v in enumerate(vals)}
            tr_codes = tr_out[c].map(mp).fillna(0).to_numpy(dtype=np.int32)
            va_codes = va_out[c].map(mp).fillna(0).to_numpy(dtype=np.int32)
            te_codes = te_out[c].map(mp).fillna(0).to_numpy(dtype=np.int32)
            n_cat = len(vals) + 1
        else:
            tr_v = tr_out[c].to_numpy(dtype=np.int64)
            va_v = va_out[c].to_numpy(dtype=np.int64)
            te_v = te_out[c].to_numpy(dtype=np.int64)
            uniq, cnt = np.unique(tr_v, return_counts=True)
            keep = uniq[cnt >= min_count]
            # Ánh xạ giá trị đủ tần suất sang 1..K, giá trị hiếm/chưa gặp về 0
            def _map_int(arr: np.ndarray) -> np.ndarray:
                if keep.size == 0:
                    return np.zeros(len(arr), dtype=np.int32)
                idx = np.searchsorted(keep, arr)
                idx_s = np.minimum(idx, keep.size - 1)
                return np.where(keep[idx_s] == arr, idx_s + 1, 0).astype(np.int32)

            tr_codes = _map_int(tr_v)
            va_codes = _map_int(va_v)
            te_codes = _map_int(te_v)
            n_cat = int(keep.size) + 1

        cats = np.arange(n_cat, dtype=np.int32)
        tr_out[c] = pd.Categorical(tr_codes, categories=cats)
        va_out[c] = pd.Categorical(va_codes, categories=cats)
        te_out[c] = pd.Categorical(te_codes, categories=cats)
        card_sizes.append(n_cat)

    return tr_out, va_out, te_out, card_sizes


class FallbackCatRealMLP(nn.Module):
    """Bản RealMLP dự phòng viết bằng PyTorch thuần khi môi trường chưa cài pytabkit."""

    def __init__(self, n_num: int, card_sizes: list[int], d_emb: int = 8):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(sz, d_emb) for sz in card_sizes])
        in_dim = n_num + len(card_sizes) * d_emb
        self.scale = nn.Parameter(torch.ones(in_dim))
        self.wide = nn.Linear(in_dim, 1)
        self.net = nn.Sequential(
            nn.Dropout(0.15),
            nn.Linear(in_dim, 384), nn.LayerNorm(384), nn.Mish(), nn.Dropout(0.25),
            nn.Linear(384, 384), nn.LayerNorm(384), nn.Mish(), nn.Dropout(0.25),
            nn.Linear(384, 256), nn.LayerNorm(256), nn.Mish(), nn.Dropout(0.20),
            nn.Linear(256, 1),
        )

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]
        x = torch.cat([x_num, *embs], dim=1) * self.scale
        return self.net(x).squeeze(1) + self.wide(x).squeeze(1)


def train_realmlp_cat_fold(
    X_tr_raw: pd.DataFrame, y_tr: np.ndarray,
    X_va_raw: pd.DataFrame, y_va: np.ndarray,
    X_te_raw: pd.DataFrame,
    cat_cols: list[str],
    device: torch.device, *, seed: int, fold_id: int,
) -> tuple[np.ndarray, np.ndarray]:
    t0 = time.perf_counter()
    X_tr_c, X_va_c, X_te_c, card_sizes = encode_fold_categories(X_tr_raw, X_va_raw, X_te_raw, cat_cols, min_count=2)

    if HAS_PYTABKIT and RealMLP_TD_Classifier is not None:
        try:
            print(f"    [RealMLP-PyTabKit F{fold_id+1}] Fitting official RealMLP_TD_Classifier (NO Tree TE, {len(cat_cols)} cat cols)...", flush=True)
            model = RealMLP_TD_Classifier(
                device=str(device),
                random_state=seed,
                n_cv=1,
                n_refit=0,
                n_epochs=REALMLP_EPOCHS,
                batch_size=REALMLP_BATCH,
                val_metric_name="cross_entropy",
                verbosity=1,
            )
            model.fit(X_tr_c, y_tr, X_val=X_va_c, y_val=y_va, cat_col_names=cat_cols)
            va_p = model.predict_proba(X_va_c)[:, 1]
            te_p = model.predict_proba(X_te_c)[:, 1]
            va_l = safe_logit(va_p).astype(np.float32)
            te_l = safe_logit(te_p).astype(np.float32)
            print(f"  --> [RealMLP-PyTabKit F{fold_id+1}] Val AUC: {roc_auc_score(y_va, va_l):.6f} | {time.perf_counter()-t0:.1f}s", flush=True)
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()
            return va_l, te_l
        except Exception as exc:
            print(f"    [RealMLP-PyTabKit F{fold_id+1}] Warning: pytabkit raised {type(exc).__name__}: {exc}. Falling back to pure-PyTorch Category-Embedding RealMLP.", flush=True)

    # Bản dự phòng PyTorch thuần nếu không có pytabkit
    print(f"    [RealMLP-Fallback F{fold_id+1}] Using pure-PyTorch Category-Embedding RealMLP (NO Tree TE)...", flush=True)
    num_cols = [c for c in X_tr_c.columns if c not in cat_cols]
    n_tr, n_va, n_te = smooth_clip_standardize(
        X_tr_c[num_cols].to_numpy(dtype=np.float32),
        X_va_c[num_cols].to_numpy(dtype=np.float32),
        X_te_c[num_cols].to_numpy(dtype=np.float32),
    )
    c_tr = np.column_stack([X_tr_c[c].cat.codes.to_numpy(dtype=np.int64) for c in cat_cols])
    c_va = np.column_stack([X_va_c[c].cat.codes.to_numpy(dtype=np.int64) for c in cat_cols])
    c_te = np.column_stack([X_te_c[c].cat.codes.to_numpy(dtype=np.int64) for c in cat_cols])

    torch.manual_seed(seed)
    fb_epochs = 3 if DRY_RUN else 12
    fb_bs = 128 if DRY_RUN else 1024
    model = FallbackCatRealMLP(n_tr.shape[1], card_sizes, d_emb=8).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=fb_epochs, eta_min=1e-5)
    xt_t, ct_t, yt_t = torch.from_numpy(n_tr).to(device), torch.from_numpy(c_tr).to(device), torch.from_numpy(y_tr.astype(np.float32)).to(device)
    xv_t, cv_t = torch.from_numpy(n_va).to(device), torch.from_numpy(c_va).to(device)
    xe_t, ce_t = torch.from_numpy(n_te).to(device), torch.from_numpy(c_te).to(device)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    top_k: list[tuple[float, dict[str, torch.Tensor]]] = []
    for ep in range(1, fb_epochs + 1):
        model.train()
        perm = torch.randperm(len(y_tr), device=device)
        for s in range(0, len(y_tr), fb_bs):
            idx = perm[s : s + fb_bs]
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                loss = F.binary_cross_entropy_with_logits(model(xt_t[idx], ct_t[idx]), yt_t[idx])
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sched.step()
        model.eval()
        with torch.inference_mode():
            va_l = torch.cat([model(xv_t[s : s + 4096], cv_t[s : s + 4096]).float() for s in range(0, len(y_va), 4096)]).cpu().numpy()
        auc = float(roc_auc_score(y_va, va_l))
        top_k.append((auc, {k_: v_.detach().cpu().clone() for k_, v_ in model.state_dict().items()}))
        top_k.sort(key=lambda t: t[0], reverse=True)
        top_k = top_k[:3]
    va_snaps, te_snaps = [], []
    for _, st in top_k:
        model.load_state_dict(st)
        model.eval()
        with torch.inference_mode():
            va_snaps.append(torch.cat([model(xv_t[s : s + 4096], cv_t[s : s + 4096]).float() for s in range(0, len(y_va), 4096)]).cpu().numpy())
            te_snaps.append(torch.cat([model(xe_t[s : s + 4096], ce_t[s : s + 4096]).float() for s in range(0, len(n_te), 4096)]).cpu().numpy())
    va_l, te_l = np.mean(va_snaps, axis=0).astype(np.float32), np.mean(te_snaps, axis=0).astype(np.float32)
    print(f"  --> [RealMLP-Fallback F{fold_id+1}] Top-3 Snapshot Val AUC: {roc_auc_score(y_va, va_l):.6f} | {time.perf_counter()-t0:.1f}s", flush=True)
    return va_l, te_l


# Phần 2: Mảng đặc trưng và WoE lồng 3 hạt giống cho ModernNCA và FT-Transformer
def extract_keys_and_plain_digits(df: pd.DataFrame) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], dict[str, list[str]]]:
    keys: dict[str, np.ndarray] = {}
    plain_d: dict[str, np.ndarray] = {}
    col_to_keys: dict[str, list[str]] = {c: [] for c in NUMERIC_COLUMNS}

    for col in NUMERIC_COLUMNS:
        c100 = np.rint(df[col].to_numpy(dtype=np.float64) * 100.0).astype(np.int64)
        k_ex = f"{col}__exact"
        keys[k_ex] = c100
        col_to_keys[col].append(k_ex)

    for col in FLOAT_COLUMNS:
        c100 = keys[f"{col}__exact"]
        ip = np.floor_divide(c100, 100)
        dec2 = np.mod(c100, 100)
        cent1 = np.mod(c100, 10)
        cent2 = np.mod(np.floor_divide(c100, 10), 10)
        unit = np.mod(ip, 10)
        tens = np.mod(np.floor_divide(ip, 10), 10)
        lead1 = _leading_digits(ip, 1)
        lead2 = _leading_digits(ip, 2)

        derived = {
            f"{col}__int": ip,
            f"{col}__r1": np.floor_divide(c100, 10),
            f"{col}__dec2": dec2,
            f"{col}__cent1": cent1,
            f"{col}__cent2": cent2,
            f"{col}__unit": unit,
            f"{col}__tens": tens,
            f"{col}__lead1": lead1,
            f"{col}__lead2": lead2,
        }
        if col in ("amount_usd", "account_balance_usd"):
            derived[f"{col}__int_mod100"] = np.mod(ip, 100)
        for kn, arr in derived.items():
            keys[kn] = arr
            col_to_keys[col].append(kn)

        # Chữ số phần lẻ dạng số nguyên
        for pn, parr in [
            (f"{col}__dec2_p", dec2),
            (f"{col}__cent1_p", cent1),
            (f"{col}__cent2_p", cent2),
            (f"{col}__unit_p", unit),
            (f"{col}__tens_p", tens),
            (f"{col}__lead1_p", lead1),
        ]:
            plain_d[pn] = parr.astype(np.float32)
            col_to_keys[col].append(pn)

    return keys, plain_d, col_to_keys


def _lookup_sorted(k_arr: np.ndarray, uniq: np.ndarray, vals: np.ndarray, default: float = 0.0) -> np.ndarray:
    idx = np.searchsorted(uniq, k_arr)
    idx_s = np.minimum(idx, uniq.size - 1)
    return np.where(uniq[idx_s] == k_arr, vals[idx_s], default).astype(np.float32)


def smooth_clip_standardize(tr: np.ndarray, va: np.ndarray, te: np.ndarray, scale_clip: float = 3.0):
    med = np.median(tr, axis=0)
    q75 = np.percentile(tr, 75, axis=0)
    q25 = np.percentile(tr, 25, axis=0)
    iqr = (q75 - q25) / 1.349
    std = tr.std(axis=0)
    s = np.where(iqr > 1e-6, iqr, np.where(std > 1e-6, std, 1.0))

    def _apply(a: np.ndarray) -> np.ndarray:
        z = (a - med) / s
        return (z / np.sqrt(1.0 + (z / scale_clip) ** 2)).astype(np.float32)

    return _apply(tr), _apply(va), _apply(te)


def build_base_arrays(df: pd.DataFrame, cat_vocabs: dict[str, dict[str, int]] | None = None):
    num = np.column_stack([df[c].to_numpy(dtype=np.float32) for c in NUMERIC_COLUMNS])
    bin_mat = np.column_stack([df[c].astype(np.float32).to_numpy() for c in BINARY_COLUMNS])

    amt = df["amount_usd"].to_numpy(dtype=np.float64)
    bal = df["account_balance_usd"].to_numpy(dtype=np.float64)
    hrs = df["hours_since_last_txn"].to_numpy(dtype=np.float64)
    dist = df["distance_from_home_km"].to_numpy(dtype=np.float64)
    hour = df["time_of_day_hour"].to_numpy(dtype=np.float64)
    dow = df["day_of_week"].to_numpy(dtype=np.float64)

    log_amt = np.log1p(np.maximum(amt, 0.0))
    log_bal = np.log1p(np.maximum(bal, 0.0))
    log_hrs = np.log1p(np.maximum(hrs, 0.0))
    log_dist = np.log1p(np.maximum(dist, 0.0))

    domain = np.column_stack([
        bin_mat.sum(axis=1), log_amt, log_bal, log_hrs, log_dist,
        log_amt - log_bal, amt / (np.maximum(bal, 0.0) + 1.0),
        (amt > bal).astype(np.float64), dist / (np.maximum(hrs, 0.0) + 0.1),
        np.sin(2.0 * np.pi * hour / 24.0), np.cos(2.0 * np.pi * hour / 24.0),
        np.sin(2.0 * np.pi * dow / 7.0), np.cos(2.0 * np.pi * dow / 7.0),
    ]).astype(np.float32)

    if cat_vocabs is None:
        cat_vocabs = {}
        for col in CATEGORICAL_COLUMNS:
            vals = sorted(df[col].dropna().astype("string").unique().tolist())
            cat_vocabs[col] = {v: i + 1 for i, v in enumerate(vals)}

    cat_ids_list, oh_parts = [], []
    for col in CATEGORICAL_COLUMNS:
        vocab = cat_vocabs[col]
        codes = df[col].astype("string").map(vocab).fillna(0).to_numpy(dtype=np.int64)
        cat_ids_list.append(codes)
        oh = np.zeros((len(df), len(vocab) + 1), dtype=np.float32)
        oh[np.arange(len(df)), codes] = 1.0
        oh_parts.append(oh)

    return num, domain, bin_mat, np.concatenate(oh_parts, axis=1), np.column_stack(cat_ids_list), cat_vocabs


def fit_transform_cross_residuals(num_tr, oh_tr, other_pairs):
    X_tr_all = np.concatenate([
        np.log1p(np.maximum(num_tr[:, [0, 1, 3, 6]], 0.0)),
        num_tr[:, [2, 4, 5, 7, 8, 9, 10, 11, 12]], oh_tr,
    ], axis=1)
    mean = X_tr_all.mean(axis=0)
    std = np.where(X_tr_all.std(axis=0) < 1e-6, 1.0, X_tr_all.std(axis=0))
    X_tr_s = (X_tr_all - mean) / std
    X_others_s = [
        (np.concatenate([np.log1p(np.maximum(n_o[:, [0, 1, 3, 6]], 0.0)), n_o[:, [2, 4, 5, 7, 8, 9, 10, 11, 12]], oh_o], axis=1) - mean) / std
        for n_o, oh_o in other_pairs
    ]
    tr_res, oth_res = [], [[] for _ in other_pairs]
    for tidx in [0, 1, 8, 11]:
        mask = np.ones(X_tr_s.shape[1], dtype=bool)
        mask[tidx] = False
        reg = Ridge(alpha=10.0).fit(X_tr_s[:, mask], X_tr_s[:, tidx])
        tr_res.append((X_tr_s[:, tidx] - reg.predict(X_tr_s[:, mask])).astype(np.float32))
        for i, xo_s in enumerate(X_others_s):
            oth_res[i].append((xo_s[:, tidx] - reg.predict(xo_s[:, mask])).astype(np.float32))
    return np.column_stack(tr_res), [np.column_stack(lst) for lst in oth_res]


def build_fold_bagged_te(
    all_keys: dict[str, np.ndarray],
    all_plain: dict[str, np.ndarray],
    test_keys: dict[str, np.ndarray],
    test_plain: dict[str, np.ndarray],
    y_all: np.ndarray,
    tr_idx: np.ndarray,
    va_idx: np.ndarray,
    *,
    fold_id: int,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], dict[str, np.ndarray]]:
    """Tính WoE lồng 5-Fold trung bình 3 hạt giống kết hợp chữ số phần lẻ (không dùng cột tần suất)."""
    y_tr = y_all[tr_idx]
    n = len(y_tr)
    tot_y = float(y_tr.sum())
    prior = tot_y / max(n, 1)
    prior_l = float(safe_logit(prior))
    seeds = (42,) if DRY_RUN else (42, 102, 202)
    inner_splits = min(5, max(2, int(min(tot_y, n - tot_y))))
    fold_sets = [
        [vi for _, vi in StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=sd + fold_id).split(np.zeros(n), y_tr)]
        for sd in seeds
    ]

    tr_d, va_d, te_d = {}, {}, {}
    for name, full_k in all_keys.items():
        k_tr, k_va, k_te = full_k[tr_idx], full_k[va_idx], test_keys[name]
        uniq, inv, tot_cnt = np.unique(k_tr, return_inverse=True, return_counts=True)
        tot_sum = np.bincount(inv, weights=y_tr, minlength=uniq.size)
        m = 5.0 if uniq.size < 2000 else 15.0
        oof_acc = np.zeros(n, dtype=np.float64)
        for f_set in fold_sets:
            for vi in f_set:
                fi = inv[vi]
                fc = np.bincount(fi, minlength=uniq.size)
                fs = np.bincount(fi, weights=y_tr[vi], minlength=uniq.size)
                oc = tot_cnt[fi] - fc[fi]
                os_ = tot_sum[fi] - fs[fi]
                fp = (tot_y - float(y_tr[vi].sum())) / max(n - len(vi), 1)
                p = (os_ + m * fp) / (oc + m)
                oof_acc[vi] += (safe_logit(p) - safe_logit(fp)) / len(seeds)
        full_woe = (safe_logit((tot_sum + m * prior) / (tot_cnt + m)) - prior_l).astype(np.float32)
        tr_d[name] = oof_acc.astype(np.float32)
        va_d[name] = _lookup_sorted(k_va, uniq, full_woe, 0.0)
        te_d[name] = _lookup_sorted(k_te, uniq, full_woe, 0.0)

    for name, arr in all_plain.items():
        tr_d[name] = arr[tr_idx]
        va_d[name] = arr[va_idx]
        te_d[name] = test_plain[name]
    return tr_d, va_d, te_d


# Mô hình 2: ModernNCA chạy trực tiếp trên bộ nhớ GPU
class PLRNumericalEmbeddings(nn.Module):
    def __init__(self, n_features: int, n_frequencies: int = 16, d_embedding: int = 8, sigma: float = 0.15):
        super().__init__()
        self.coefficients = nn.Parameter(torch.randn(n_features, n_frequencies) * sigma)
        self.proj_w = nn.Parameter(torch.empty(n_features, 2 * n_frequencies, d_embedding))
        self.proj_b = nn.Parameter(torch.zeros(n_features, d_embedding))
        nn.init.kaiming_uniform_(self.proj_w, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        angles = 2.0 * math.pi * x.unsqueeze(-1) * self.coefficients.unsqueeze(0)
        per = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        out = torch.einsum("bfm,fmd->bfd", per, self.proj_w) + self.proj_b.unsqueeze(0)
        return F.silu(out).flatten(1)


class ModernNCAEncoder(nn.Module):
    def __init__(self, n_per: int, n_oth: int, d_hidden: int = 256, d_embed: int = 128, dropout: float = 0.2):
        super().__init__()
        self.plr = PLRNumericalEmbeddings(n_per, n_frequencies=16, d_embedding=8, sigma=0.15)
        in_dim = n_per * 8 + n_oth
        self.in_scale = nn.Parameter(torch.ones(in_dim))
        self.in_proj = nn.Linear(in_dim, d_hidden)
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_hidden), nn.Linear(d_hidden, d_hidden), nn.SiLU(), nn.Dropout(dropout),
                nn.Linear(d_hidden, d_hidden), nn.Dropout(dropout),
            )
            for _ in range(2)
        ])
        self.out_proj = nn.Sequential(nn.LayerNorm(d_hidden), nn.SiLU(), nn.Linear(d_hidden, d_embed))
        self.log_scale = nn.Parameter(torch.tensor(0.0))

    def forward(self, x_per: torch.Tensor, x_oth: torch.Tensor) -> torch.Tensor:
        x = torch.cat([self.plr(x_per), x_oth], dim=1) * self.in_scale
        h = self.in_proj(x)
        for blk in self.blocks:
            h = h + blk(h)
        z = self.out_proj(h)
        return F.normalize(z, p=2, dim=1) * (math.sqrt(z.size(1)) * 0.5)


def nca_soft_nn_logits(q_emb, pos_emb, neg_emb, prior_logit, inv_temp, mask_self_pos, mask_self_neg):
    q_norm = (q_emb * q_emb).sum(dim=1, keepdim=True)
    p_norm = (pos_emb * pos_emb).sum(dim=1).unsqueeze(0)
    n_norm = (neg_emb * neg_emb).sum(dim=1).unsqueeze(0)
    scale = inv_temp.clamp(0.05, 5.0)
    s_pos = -scale * (q_norm + p_norm - 2.0 * (q_emb @ pos_emb.T)).clamp_min(0.0)
    s_neg = -scale * (q_norm + n_norm - 2.0 * (q_emb @ neg_emb.T)).clamp_min(0.0)
    s_pos = s_pos.masked_fill(mask_self_pos, -1e9)
    s_neg = s_neg.masked_fill(mask_self_neg, -1e9)
    eff_p = (pos_emb.size(0) - mask_self_pos.float().sum(dim=1)).clamp_min(1.0)
    eff_n = (neg_emb.size(0) - mask_self_neg.float().sum(dim=1)).clamp_min(1.0)
    l_pos = torch.logsumexp(s_pos, dim=1) - torch.log(eff_p)
    l_neg = torch.logsumexp(s_neg, dim=1) - torch.log(eff_n)
    return prior_logit + (l_pos - l_neg)


def train_modernnca_fold(
    per_tr, oth_tr, y_tr, per_va, oth_va, y_va, per_te, oth_te, device, *, seed: int, fold_id: int
) -> tuple[np.ndarray, np.ndarray]:
    t0 = time.perf_counter()
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    prior_logit = float(safe_logit(float(y_tr.mean())))
    pos_idx_np = np.where(y_tr == 1)[0]
    neg_idx_np = np.where(y_tr == 0)[0]
    eval_neg_idx = torch.from_numpy(rng.choice(neg_idx_np, size=min(len(neg_idx_np), NCA_EVAL_NEG_SUP), replace=False)).to(device)

    # Giữ tensor của từng fold trên VRAM để truy hồi nhanh
    p_tr_t = torch.from_numpy(per_tr).to(device)
    o_tr_t = torch.from_numpy(oth_tr).to(device)
    y_tr_t = torch.from_numpy(y_tr.astype(np.float32)).to(device)
    pos_idx_t = torch.from_numpy(pos_idx_np).to(device)
    neg_idx_t = torch.from_numpy(neg_idx_np).to(device)
    p_va_t, o_va_t = torch.from_numpy(per_va).to(device), torch.from_numpy(oth_va).to(device)
    p_te_t, o_te_t = torch.from_numpy(per_te).to(device), torch.from_numpy(oth_te).to(device)

    model = ModernNCAEncoder(per_tr.shape[1], oth_tr.shape[1], d_hidden=256, d_embed=128, dropout=0.2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1.4e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NCA_EPOCHS, eta_min=1e-5)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    @torch.inference_mode()
    def encode_idx(pt, ot, idx=None):
        model.eval()
        n_rows = len(pt) if idx is None else len(idx)
        chunks = []
        for s in range(0, n_rows, 8192):
            if idx is None:
                pb, ob = pt[s : s + 8192], ot[s : s + 8192]
            else:
                sl = idx[s : s + 8192]
                pb, ob = pt[sl], ot[sl]
            with torch.amp.autocast("cuda", enabled=use_amp):
                chunks.append(model(pb, ob).float())
        return torch.cat(chunks, dim=0)

    @torch.inference_mode()
    def predict_retrieval(q_pt, q_ot, pos_bank, neg_bank):
        model.eval()
        scale = torch.exp(model.log_scale).float().clamp(0.05, 5.0)
        p_norm = (pos_bank * pos_bank).sum(dim=1).unsqueeze(0)
        n_norm = (neg_bank * neg_bank).sum(dim=1).unsqueeze(0)
        log_mp, log_mn = math.log(pos_bank.size(0)), math.log(neg_bank.size(0))
        out = []
        for s in range(0, len(q_pt), 2048):
            with torch.amp.autocast("cuda", enabled=use_amp):
                qe = model(q_pt[s : s + 2048], q_ot[s : s + 2048]).float()
            qn = (qe * qe).sum(dim=1, keepdim=True)
            dp = (qn + p_norm - 2.0 * (qe @ pos_bank.T)).clamp_min(0.0)
            lse_pos = torch.logsumexp(-scale * dp, dim=1) - log_mp
            lse_neg_acc = None
            for ns in range(0, neg_bank.size(0), 16384):
                nb = neg_bank[ns : ns + 16384]
                dn = (qn + n_norm[:, ns : ns + 16384] - 2.0 * (qe @ nb.T)).clamp_min(0.0)
                cur = torch.logsumexp(-scale * dn, dim=1)
                lse_neg_acc = cur if lse_neg_acc is None else torch.logaddexp(lse_neg_acc, cur)
            out.append((prior_logit + (lse_pos - (lse_neg_acc - log_mn))).cpu().numpy())
        return np.concatenate(out, axis=0).astype(np.float32)

    n = len(y_tr)
    n_pos_tot, n_neg_tot = len(pos_idx_t), len(neg_idx_t)
    top_k: list[tuple[float, int, dict[str, torch.Tensor]]] = []
    best_auc, best_ep = -1.0, 0

    for ep in range(1, NCA_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n, device=device)
        for s in range(0, n, NCA_BATCH):
            q_idx = perm[s : s + NCA_BATCH]
            sp_idx = pos_idx_t[torch.randint(0, n_pos_tot, (NCA_TRAIN_POS_SUP,), device=device)]
            sn_idx = neg_idx_t[torch.randint(0, n_neg_tot, (NCA_TRAIN_NEG_SUP,), device=device)]

            mask_p = q_idx.unsqueeze(1) == sp_idx.unsqueeze(0)
            mask_n = q_idx.unsqueeze(1) == sn_idx.unsqueeze(0)

            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                q_e = model(p_tr_t[q_idx], o_tr_t[q_idx]).float()
                sp_e = model(p_tr_t[sp_idx], o_tr_t[sp_idx]).float()
                sn_e = model(p_tr_t[sn_idx], o_tr_t[sn_idx]).float()
                logits = nca_soft_nn_logits(q_e, sp_e, sn_e, prior_logit, torch.exp(model.log_scale).float(), mask_p, mask_n)
                loss = F.binary_cross_entropy_with_logits(logits, y_tr_t[q_idx])
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        sched.step()

        pos_bank = encode_idx(p_tr_t, o_tr_t, pos_idx_t)
        neg_bank = encode_idx(p_tr_t, o_tr_t, eval_neg_idx)
        va_l = predict_retrieval(p_va_t, o_va_t, pos_bank, neg_bank)
        auc = float(roc_auc_score(y_va, va_l))
        st = {k_: v_.detach().cpu().clone() for k_, v_ in model.state_dict().items()}
        top_k.append((auc, ep, st))
        top_k.sort(key=lambda t: t[0], reverse=True)
        top_k = top_k[:2]
        if auc > best_auc:
            best_auc, best_ep = auc, ep
        if ep == 1 or ep == best_ep or ep % 4 == 0:
            print(f"    [ModernNCA F{fold_id+1}] Ep {ep:02d} | Val AUC: {auc:.6f} (best={best_auc:.6f} @ ep {best_ep})", flush=True)
        if ep - best_ep >= NCA_PATIENCE:
            break

    va_snaps, te_snaps = [], []
    for _, _, st in top_k:
        model.load_state_dict(st)
        pos_bank = encode_idx(p_tr_t, o_tr_t, pos_idx_t)
        neg_bank = encode_idx(p_tr_t, o_tr_t, eval_neg_idx)
        va_snaps.append(predict_retrieval(p_va_t, o_va_t, pos_bank, neg_bank))
        te_snaps.append(predict_retrieval(p_te_t, o_te_t, pos_bank, neg_bank))
    va_out, te_out = np.mean(va_snaps, axis=0).astype(np.float32), np.mean(te_snaps, axis=0).astype(np.float32)
    print(f"  --> [ModernNCA F{fold_id+1}] Top-2 Snapshot AUC: {roc_auc_score(y_va, va_out):.6f} | {time.perf_counter()-t0:.1f}s", flush=True)
    del p_tr_t, o_tr_t, y_tr_t, p_va_t, o_va_t, p_te_t, o_te_t, model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return va_out, te_out


# Mô hình 3: FT-Transformer 24 trường trên GPU
class GEGLUBlock(nn.Module):
    def __init__(self, d_token: int, n_heads: int = 4, ffn_mult: float = 2.0, dropout: float = 0.15):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.attn = nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_token)
        hidden = int(d_token * ffn_mult)
        self.fc1 = nn.Linear(d_token, hidden * 2)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, d_token)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = self.norm1(x)
        attn_out, _ = self.attn(u, u, u, need_weights=False)
        x = x + attn_out
        u2 = self.norm2(x)
        a, b = self.fc1(u2).chunk(2, dim=-1)
        return x + self.drop(self.fc2(self.drop(a * F.gelu(b))))


class FieldFTTransformer(nn.Module):
    def __init__(self, field_dims: list[int], cat_sizes: list[int], d_token: int = 64, n_layers: int = 3, n_heads: int = 4, dropout: float = 0.15):
        super().__init__()
        self.field_projs = nn.ModuleList([nn.Sequential(nn.Linear(d, d_token), nn.SiLU(), nn.Linear(d_token, d_token)) for d in field_dims])
        self.cat_embs = nn.ModuleList([nn.Embedding(sz, d_token, padding_idx=0) for sz in cat_sizes])
        n_tokens = 1 + len(field_dims) + len(cat_sizes)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)
        self.field_pos = nn.Parameter(torch.randn(1, n_tokens, d_token) * 0.02)
        self.layers = nn.ModuleList([GEGLUBlock(d_token, n_heads=n_heads, ffn_mult=2.0, dropout=dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.SiLU(), nn.Linear(d_token, 1))

    def forward(self, field_tensors: list[torch.Tensor], cat_ids: torch.Tensor) -> torch.Tensor:
        B = cat_ids.size(0)
        tokens = [self.cls_token.expand(B, -1, -1)]
        for proj, ft in zip(self.field_projs, field_tensors):
            tokens.append(proj(ft).unsqueeze(1))
        for i, emb in enumerate(self.cat_embs):
            tokens.append(emb(cat_ids[:, i]).unsqueeze(1))
        x = torch.cat(tokens, dim=1) + self.field_pos
        for layer in self.layers:
            x = layer(x)
        return self.head(x[:, 0]).squeeze(1)


def build_field_groups(num_s, dom_s, res_s, bin_mat, te_dict, col_to_keys) -> list[np.ndarray]:
    fields = []
    for i, col in enumerate(NUMERIC_COLUMNS):
        parts = [num_s[:, i : i + 1], *[te_dict[kn][:, None] for kn in col_to_keys[col]]]
        fields.append(np.concatenate(parts, axis=1).astype(np.float32))
    fields.append(np.concatenate([dom_s, res_s], axis=1).astype(np.float32))
    for j in range(bin_mat.shape[1]):
        fields.append(bin_mat[:, j : j + 1].astype(np.float32))
    return fields


def train_ft_transformer_fold(
    fields_tr, cat_tr, y_tr, fields_va, cat_va, y_va, fields_te, cat_te, cat_sizes, device, *, seed: int, fold_id: int
) -> tuple[np.ndarray, np.ndarray]:
    t0 = time.perf_counter()
    torch.manual_seed(seed)
    field_dims = [f.shape[1] for f in fields_tr]
    X_tr = torch.from_numpy(np.concatenate(fields_tr, axis=1)).to(device)
    X_va = torch.from_numpy(np.concatenate(fields_va, axis=1)).to(device)
    X_te = torch.from_numpy(np.concatenate(fields_te, axis=1)).to(device)
    C_tr, C_va, C_te = torch.from_numpy(cat_tr).to(device), torch.from_numpy(cat_va).to(device), torch.from_numpy(cat_te).to(device)
    Y_tr = torch.from_numpy(y_tr.astype(np.float32)).to(device)

    model = FieldFTTransformer(field_dims, cat_sizes, d_token=64, n_layers=3, n_heads=4, dropout=0.15).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1.2e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FTT_EPOCHS, eta_min=1e-5)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    @torch.inference_mode()
    def predict(Xt, Ct):
        model.eval()
        out = []
        for s in range(0, len(Xt), 4096):
            with torch.amp.autocast("cuda", enabled=use_amp):
                out.append(model(list(torch.split(Xt[s : s + 4096], field_dims, dim=1)), Ct[s : s + 4096]).float().cpu().numpy())
        return np.concatenate(out, axis=0)

    top_k: list[tuple[float, int, dict[str, torch.Tensor]]] = []
    best_auc, best_ep = -1.0, 0
    n = len(y_tr)

    for ep in range(1, FTT_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n, device=device)
        for s in range(0, n, FTT_BATCH):
            idx = perm[s : s + FTT_BATCH]
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                loss = F.binary_cross_entropy_with_logits(model(list(torch.split(X_tr[idx], field_dims, dim=1)), C_tr[idx]), Y_tr[idx])
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        sched.step()

        va_l = predict(X_va, C_va)
        auc = float(roc_auc_score(y_va, va_l))
        st = {k_: v_.detach().cpu().clone() for k_, v_ in model.state_dict().items()}
        top_k.append((auc, ep, st))
        top_k.sort(key=lambda t: t[0], reverse=True)
        top_k = top_k[:3]
        if auc > best_auc:
            best_auc, best_ep = auc, ep
        if ep == 1 or ep == best_ep or ep % 5 == 0:
            print(f"    [FT-Trans F{fold_id+1}] Ep {ep:02d} | Val AUC: {auc:.6f} (best={best_auc:.6f} @ ep {best_ep})", flush=True)
        if ep - best_ep >= FTT_PATIENCE:
            break

    va_snaps, te_snaps = [], []
    for _, _, st in top_k:
        model.load_state_dict(st)
        va_snaps.append(predict(X_va, C_va))
        te_snaps.append(predict(X_te, C_te))
    va_out, te_out = np.mean(va_snaps, axis=0).astype(np.float32), np.mean(te_snaps, axis=0).astype(np.float32)
    print(f"  --> [FT-Trans F{fold_id+1}] Top-3 Snapshot AUC: {roc_auc_score(y_va, va_out):.6f} | {time.perf_counter()-t0:.1f}s", flush=True)
    del X_tr, X_va, X_te, C_tr, C_va, C_te, Y_tr, model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return va_out, te_out


def main():
    t_start = time.perf_counter()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data_dir = locate_data_dir()
    out_dir = locate_output_dir()
    print(f"Device: {device} | pytabkit={HAS_PYTABKIT} | Data: {data_dir} | Output: {out_dir} | DRY_RUN={DRY_RUN}", flush=True)

    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")
    sample_sub = pd.read_csv(data_dir / "sample_submission.csv", usecols=[ID_COLUMN])

    if DRY_RUN:
        pos = train_df[train_df[TARGET_COLUMN] == 1].head(300)
        neg = train_df[train_df[TARGET_COLUMN] == 0].head(2700)
        train_df = pd.concat([pos, neg], ignore_index=True)
        test_df = test_df.head(500).reset_index(drop=True)
        sample_sub = pd.DataFrame({ID_COLUMN: test_df[ID_COLUMN]})

    y_all = train_df[TARGET_COLUMN].to_numpy(dtype=np.int64)
    train_ids = train_df[ID_COLUMN].to_numpy()
    test_ids = test_df[ID_COLUMN].to_numpy()
    sample_ids = pd.Index(sample_sub[ID_COLUMN])

    rmlp_tr_all, rmlp_te_all, rmlp_cat_cols = build_realmlp_no_te_tables(train_df, test_df)
    num_tr_all, dom_tr_all, bin_tr_all, oh_tr_all, cat_tr_all, vocabs = build_base_arrays(train_df)
    num_te_all, dom_te_all, bin_te_all, oh_te_all, cat_te_all, _ = build_base_arrays(test_df, cat_vocabs=vocabs)
    cat_sizes = [len(vocabs[c]) + 1 for c in CATEGORICAL_COLUMNS]

    all_keys, all_plain, col_to_keys = extract_keys_and_plain_digits(train_df)
    test_keys, test_plain, _ = extract_keys_and_plain_digits(test_df)
    te_key_order = list(all_keys.keys()) + list(all_plain.keys())

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_rmlp = np.zeros(len(train_df), dtype=np.float64)
    test_rmlp = np.zeros(len(test_df), dtype=np.float64)
    oof_nca = np.zeros(len(train_df), dtype=np.float64)
    test_nca = np.zeros(len(test_df), dtype=np.float64)
    oof_ftt = np.zeros(len(train_df), dtype=np.float64)
    test_ftt = np.zeros(len(test_df), dtype=np.float64)

    for fold_id, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
        t_f = time.perf_counter()
        print(f"\n{'=' * 72}\nFOLD {fold_id + 1}/{N_SPLITS} (Train={len(tr_idx):,} | Val={len(va_idx):,})\n{'=' * 72}", flush=True)
        y_tr, y_va = y_all[tr_idx], y_all[va_idx]

        # 1. Huấn luyện RealMLP_TD không dùng Target Encoding
        v1, t1 = train_realmlp_cat_fold(
            rmlp_tr_all.iloc[tr_idx].reset_index(drop=True), y_tr,
            rmlp_tr_all.iloc[va_idx].reset_index(drop=True), y_va,
            rmlp_te_all, rmlp_cat_cols, device, seed=SEED + fold_id, fold_id=fold_id,
        )
        oof_rmlp[va_idx] = v1
        test_rmlp += t1 / N_SPLITS

        # Đặc trưng dùng chung cho ModernNCA và FT-Transformer
        res_tr, [res_va, res_te] = fit_transform_cross_residuals(
            num_tr_all[tr_idx], oh_tr_all[tr_idx],
            [(num_tr_all[va_idx], oh_tr_all[va_idx]), (num_te_all, oh_te_all)],
        )
        te_tr_d, te_va_d, te_te_d = build_fold_bagged_te(
            all_keys, all_plain, test_keys, test_plain, y_all, tr_idx, va_idx, fold_id=fold_id
        )
        num_tr_s, num_va_s, num_te_s = smooth_clip_standardize(num_tr_all[tr_idx], num_tr_all[va_idx], num_te_all)
        dom_tr_s, dom_va_s, dom_te_s = smooth_clip_standardize(dom_tr_all[tr_idx], dom_tr_all[va_idx], dom_te_all)
        res_tr_s, res_va_s, res_te_s = smooth_clip_standardize(res_tr, res_va, res_te)

        te_tr_s, te_va_s, te_te_s = smooth_clip_standardize(
            np.column_stack([te_tr_d[k] for k in te_key_order]),
            np.column_stack([te_va_d[k] for k in te_key_order]),
            np.column_stack([te_te_d[k] for k in te_key_order]),
            scale_clip=4.0,
        )
        te_tr_sd = {k: te_tr_s[:, i] for i, k in enumerate(te_key_order)}
        te_va_sd = {k: te_va_s[:, i] for i, k in enumerate(te_key_order)}
        te_te_sd = {k: te_te_s[:, i] for i, k in enumerate(te_key_order)}

        per_tr = np.concatenate([num_tr_s, res_tr_s], axis=1)
        per_va = np.concatenate([num_va_s, res_va_s], axis=1)
        per_te = np.concatenate([num_te_s, res_te_s], axis=1)
        oth_tr = np.concatenate([dom_tr_s, bin_tr_all[tr_idx], oh_tr_all[tr_idx], te_tr_s], axis=1)
        oth_va = np.concatenate([dom_va_s, bin_tr_all[va_idx], oh_tr_all[va_idx], te_va_s], axis=1)
        oth_te = np.concatenate([dom_te_s, bin_te_all, oh_te_all, te_te_s], axis=1)

        # 2. Huấn luyện ModernNCA
        v2, t2 = train_modernnca_fold(per_tr, oth_tr, y_tr, per_va, oth_va, y_va, per_te, oth_te, device, seed=200 + SEED + fold_id, fold_id=fold_id)
        oof_nca[va_idx] = v2
        test_nca += t2 / N_SPLITS

        # 3. Huấn luyện FT-Transformer 24 trường
        f_tr = build_field_groups(num_tr_s, dom_tr_s, res_tr_s, bin_tr_all[tr_idx], te_tr_sd, col_to_keys)
        f_va = build_field_groups(num_va_s, dom_va_s, res_va_s, bin_tr_all[va_idx], te_va_sd, col_to_keys)
        f_te = build_field_groups(num_te_s, dom_te_s, res_te_s, bin_te_all, te_te_sd, col_to_keys)
        v3, t3 = train_ft_transformer_fold(
            f_tr, cat_tr_all[tr_idx], y_tr, f_va, cat_tr_all[va_idx], y_va, f_te, cat_te_all,
            cat_sizes, device, seed=400 + SEED + fold_id, fold_id=fold_id,
        )
        oof_ftt[va_idx] = v3
        test_ftt += t3 / N_SPLITS

        print(f"  [FOLD {fold_id+1} COMPLETE in {(time.perf_counter()-t_f)/60:.1f} min]", flush=True)
        gc.collect()

    auc_rmlp = float(roc_auc_score(y_all, oof_rmlp))
    auc_nca = float(roc_auc_score(y_all, oof_nca))
    auc_ftt = float(roc_auc_score(y_all, oof_ftt))
    oof_blend = 0.45 * oof_rmlp + 0.30 * oof_nca + 0.25 * oof_ftt
    test_blend = 0.45 * test_rmlp + 0.30 * test_nca + 0.25 * test_ftt
    auc_blend = float(roc_auc_score(y_all, oof_blend))
    rec_blend = float(recall_score(y_all, expit(oof_blend) >= 0.5, zero_division=0))

    print(f"\n{'=' * 72}")
    print(f"RealMLP-Cat (NO Tree TE) {N_SPLITS}-Fold OOF AUC : {auc_rmlp:.6f}")
    print(f"ModernNCA (ICLR 2025)    {N_SPLITS}-Fold OOF AUC : {auc_nca:.6f}")
    print(f"24-Field FT-Transformer  {N_SPLITS}-Fold OOF AUC : {auc_ftt:.6f}")
    print(f"3-Architecture Blend OOF AUC           : {auc_blend:.6f} | Recall@0.5: {rec_blend:.6f}")
    print(f"Total elapsed: {(time.perf_counter() - t_start) / 60:.1f} min")
    print(f"{'=' * 72}", flush=True)

    pd.DataFrame({
        ID_COLUMN: train_ids,
        TARGET_COLUMN: y_all,
        "pred_prob": expit(oof_blend),
        "logit_realmlp_cat": oof_rmlp,
        "logit_modernnca": oof_nca,
        "logit_fttrans": oof_ftt,
    }).to_csv(out_dir / "oof_kaggle_sota2025_v2_5fold.csv", index=False)

    sub_df = pd.DataFrame({ID_COLUMN: sample_ids})
    sub_df[TARGET_COLUMN] = pd.Series(expit(test_blend), index=test_ids).reindex(sample_ids).to_numpy()
    sub_df.to_csv(out_dir / "submission.csv", index=False)
    sub_df.to_csv(out_dir / "sub_kaggle_sota2025_v2_5fold.csv", index=False)

    det_df = pd.DataFrame({ID_COLUMN: sample_ids})
    for col_name, arr in [("logit_realmlp_cat", test_rmlp), ("logit_modernnca", test_nca), ("logit_fttrans", test_ftt)]:
        det_df[col_name] = pd.Series(arr, index=test_ids).reindex(sample_ids).to_numpy()
    det_df.to_csv(out_dir / "sub_kaggle_sota2025_v2_details.csv", index=False)
    print(f"Saved submission.csv, oof_kaggle_sota2025_v2_5fold.csv, and sub_kaggle_sota2025_v2_details.csv in {out_dir}", flush=True)


In [ ]:
main()
